# D: continuous angle commands / непрерывное задание угла

The input is the desired angle in degrees. The current polynomials, neurons, muscles, full afferents and limb evolve continuously. No PD/PID controller or hidden-state reset is used between commands. See [README](README.md) / [English](README.en.md). Training is never launched by this notebook.

Вход — желаемый угол, фактическое движение может отставать. Все графики ниже вычисляются из сохранённого генома; начальные состояния и параметры сигнала можно менять.

<!-- current-training -->
**Переобучение: времена мышц ≥50 мс.** Геном `C_transfer`. Завершено тестов 14/18, строгий успех 0/18. Перезапустите ядро. [Результаты](results/README.md).

**English.** Muscle times ≥50 ms; winner `C_transfer`. Completed 14/18; strict success 0/18. Restart the kernel. [Results](results/README.en.md).


**Проверка этой серии:** резкий переход цели +45° → −45° вызвал падение через 0,5864 с. После остановки используйте «Сброс».

**English:** the +45° → −45° target step caused a fall after 0.5864 s in this study; use Reset after the simulation stops.


<!-- mirror-genome-variants -->

Три сохранённые версии: [быстрые мышцы](best_genome_fast_muscle.json), [времена ≥50 мс](best_genome_slow_muscle.json), [времена ≥50 мс и симметрия](best_genome_slow_muscle_mirrow.json). Выберите версию через `GENOME_VARIANT`; `GENOME_PATH` позволяет указать другой файл. По умолчанию выбран зеркальный геном. Повторный запуск ячейки обновляет старые загрузчики в памяти ядра. [Происхождение и параметры](GENOME_VERSIONS.md).

Three saved variants: fast muscles, ≥50 ms muscles, and ≥50 ms with exact mirror symmetry. Select a named version using `GENOME_VARIANT`, or override it with `GENOME_PATH`. The mirror genome is selected by default. Rerunning the cell refreshes stale cached codecs. [Sources and values](GENOME_VERSIONS.en.md).

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager


def locate_project():
    """Find the experiment project from a notebook or repository working directory."""
    for parent in (Path.cwd(), *Path.cwd().parents):
        for candidate in (parent, parent / "scripts/AfferentBalance2026"):
            if (candidate / "experiments/D_trajectory/genome_cfg.json").is_file():
                return candidate.resolve()
    raise FileNotFoundError("Cannot locate AfferentBalance2026")

PROJECT = locate_project()
sys.path.insert(0, str(PROJECT.parents[1]))
import importlib
from scripts.AfferentBalance2026.experiments import notebook_runtime as _notebook_runtime
# Refresh this helper too: an open kernel may retain its previous implementation.
importlib.reload(_notebook_runtime)
# Update core.genome before importing dependants that require MIRROR_INDICES.
_notebook_runtime.refresh_model_modules()
resolve_genome_path = _notebook_runtime.resolve_genome_path
from scripts.AfferentBalance2026.experiments.D_trajectory.controller import AngleController
from scripts.AfferentBalance2026.experiments.D_trajectory.controller import load_solution, TrajectoryGenome
from scripts.AfferentBalance2026.experiments.D_trajectory.trajectory import replay, sine_case, command_currents
from scripts.AfferentBalance2026.experiments.simulation import MODEL
if MODEL != "SpikingNN.Afferents/v3-tensile":
    raise RuntimeError("Restart the Jupyter kernel: tensile-torque model v3 is required.")

for directory in (Path.home() / ".local/share/fonts", Path.home() / ".wine/drive_c/windows/Fonts"):
    for path in directory.rglob("times*.ttf"):
        font_manager.fontManager.addfont(str(path))
plt.rcParams.update({"font.family": "Times New Roman", "font.size": 14, "pdf.fonttype": 42})
SMOKE = os.environ.get("SPIKINGNN_NOTEBOOK_SMOKE") == "1"
FOLDER = PROJECT / "experiments/D_trajectory"
# Choose fast_muscle, slow_muscle, slow_muscle_mirrow, or current.
GENOME_VARIANT = "slow_muscle_mirrow"
GENOME_PATH = None  # Optional explicit path; a bare filename belongs to FOLDER.
GENOME_PATH = resolve_genome_path(FOLDER, GENOME_PATH, variant=GENOME_VARIANT)
cfg, physical, coefficients, tau_s, metadata = load_solution(GENOME_PATH)
print(GENOME_PATH, "current filter (s):", tau_s)

## Stateful input / подача угла
`advance(angle, duration_s)` holds the command for a block with a 0.1 ms internal step. Assigning `target_angle_deg` changes only the command. `reset` is explicit.

In [ ]:
controller = AngleController(GENOME_PATH, initial_angle_deg=-30, target_angle_deg=0)
state = controller.advance(15, duration_s=0.01)
controller.target_angle_deg = 10
state = controller.step()
print({key: state[key] for key in ("time_s", "target_angle_deg", "angle_deg", "input_currents")})

In [ ]:
def run_sine(amplitude_deg=5, period_s=50, initial_angle_deg=0, initial_speed_rad_s=0,
             p=None, c=None):
    """Replay one continuous sine command; retain every 0.1 ms sample."""
    spec = sine_case(amplitude_deg, period_s, initial_angle=initial_angle_deg,
                     initial_speed=initial_speed_rad_s)
    if SMOKE:
        spec.update(duration_s=0.02, hold_s=0.002, ramp_s=0.002)
    result = replay(physical if p is None else p, coefficients if c is None else c,
                    cfg.mechanics, spec, tau_s=tau_s)
    return spec, result


def view_states(result):
    """Show angle, currents, neurons, muscles, afferents and the actual phase trajectory."""
    tr, ref, currents = result["trace"], result["reference"], result["commands"]
    sample = np.unique(np.r_[np.arange(0, len(tr), 1 if SMOKE else 50), len(tr)-1])
    t = tr[sample, 0]
    fig, axes = plt.subplots(4, 2, figsize=(15, 17), layout="constrained")
    axes[0, 0].plot(t, np.rad2deg(tr[sample, 1]), label="Limb")
    axes[0, 0].plot(t, np.rad2deg(ref[sample, 0]), "k--", label="Target")
    axes[0, 1].plot(np.rad2deg(tr[sample, 1]), tr[sample, 2], label="Limb")
    axes[0, 1].plot(np.rad2deg(ref[sample, 0]), ref[sample, 1], "k--", label="Target")
    axes[1, 0].plot(t, currents[sample, :2], label=["Neuron 0", "Neuron 1"])
    window = np.arange(len(tr)) < 3000
    axes[1, 1].plot(tr[sample, 0], tr[sample, 5:7], label=["Neuron 0", "Neuron 1"], lw=.8)
    axes[2, 0].plot(t, tr[sample, 3:5], label=["Flexor", "Extensor"])
    axes[2, 1].plot(t, tr[sample, 17:19], label=["Flexor", "Extensor"])
    axes[3, 0].plot(t, tr[sample, 11:14], label=["Ia", "II", "Ib"])
    axes[3, 1].plot(t, tr[sample, 14:17], label=["Ia", "II", "Ib"])
    labels = [("Limb", "Angle (deg)"), ("Phase trajectory", "Velocity (rad/s)"),
              ("Applied currents", "Current (model units)"), ("Membrane potentials", "mV"),
              ("Muscle forces", "N"), ("Hill activation", "Activation"),
              ("Flexor afferents", "Response"), ("Extensor afferents", "Response")]
    for i, (ax, (title, ylabel)) in enumerate(zip(axes.ravel(), labels)):
        ax.set(title=title, xlabel="Angle (deg)" if i == 1 else "Time (s)", ylabel=ylabel)
        ax.legend()
        ax.grid(alpha=.25)
    return fig

spec, result = run_sine()
print("Tracking RMSE (deg):", np.rad2deg(result["metrics"][2]), "failure:", result["failure"])
fig = view_states(result)

In [ ]:
def phase_grid(angles_deg=(-45, 0, 45), speeds_rad_s=(-0.5, 0.5), amplitude_deg=30, period_s=20):
    """Simulate distinct initial conditions under the same moving angle command."""
    fig, ax = plt.subplots(figsize=(10, 7))
    rows = []
    for theta in angles_deg:
        for omega in speeds_rad_s:
            _, out = run_sine(amplitude_deg, period_s, theta, omega)
            full_trace = out["trace"]
            indices = np.unique(np.r_[np.arange(0, len(full_trace), 1 if SMOKE else 100), len(full_trace)-1])
            tr = full_trace[indices]
            ax.plot(np.rad2deg(tr[:, 1]), tr[:, 2], lw=.8, label=f"{theta:g}°, {omega:g} rad/s")
            if out["failure"]:
                ax.plot(np.rad2deg(tr[-1, 1]), tr[-1, 2], "kx", ms=7)
            rows.append((theta, omega, np.rad2deg(out["metrics"][2]), out["failure"]))
    ax.set(xlabel="Angle (deg)", ylabel="Angular velocity (rad/s)", title="Continuous tracking from different initial states")
    ax.legend(fontsize=11)
    ax.grid(alpha=.25)
    return fig, rows

# For a 10-by-10 grid: phase_grid(np.linspace(-50, 50, 10), np.linspace(-1, 1, 10)).
fig, grid_metrics = phase_grid(angles_deg=(0,) if SMOKE else (-45, 0, 45), speeds_rad_s=(0,) if SMOKE else (-.5, .5))
grid_metrics

In [ ]:
def scan_parameter(name, values):
    """Scan a physical parameter, preserving fixed values and optional mirror pairs."""
    from scripts.AfferentBalance2026.experiments.core.genome import MIRROR_INDICES
    index = next(i for i, field in enumerate(cfg.data["parameters"]) if field["name"] == name)
    if index not in cfg.mutable:
        raise ValueError("This parameter is fixed by the genome configuration")
    rows = []
    for value in values:
        p = physical.copy()
        p[index] = value
        if cfg.mirror_symmetry:
            p[MIRROR_INDICES[index]] = value  # Preserve the reflected physical pair.
        cfg.encode(p)
        _, out = run_sine(p=p)
        rows.append((value, np.rad2deg(out["metrics"][2]), out["failure"]))
    return np.asarray(rows)

# Example: scan_parameter("muscles.flexor.tau_c", [50, 71, 100]).
# Input-current curve already encoded in this genome:
targets = np.linspace(-45, 45, 181)
commands = np.array([command_currents(coefficients, np.deg2rad(angle)) for angle in targets])
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(targets, commands, label=["Neuron 0", "Neuron 1"])
ax.set(xlabel="Desired angle (deg)", ylabel="Current (model units)", title="Genome current law")
ax.legend()
ax.grid(alpha=.25)

In [ ]:
# LIVE_ANGLE_SIMULATION: run this cell on its own; earlier cells are not required.
# All UI code, configuration, class definition and launch are contained here.
# The existing AngleController supplies the tested neural/muscle/Afferents physics.
import asyncio
from collections import deque
from datetime import datetime
from html import escape
from io import BytesIO
import math
import os
from pathlib import Path
import sys
import time

import ipywidgets as widgets
from IPython.display import display
import matplotlib as mpl
from matplotlib import font_manager
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.figure import Figure
from matplotlib.patches import FancyArrowPatch
import numpy as np

# ---------------- USER SETTINGS: change these and rerun this cell ----------------
# Choose fast_muscle, slow_muscle, slow_muscle_mirrow, or current.
GENOME_VARIANT = "slow_muscle_mirrow"
GENOME_PATH = None              # Optional explicit path; overrides GENOME_VARIANT.
INITIAL_ANGLE_DEG = 0.0         # Actual initial angle relative to the unstable vertical.
INITIAL_SPEED_RAD_S = 0.0       # Actual initial angular velocity.
INITIAL_TARGET_DEG = 0.0        # Slider starts at this desired angle, within [-45, 45].
TARGET_STEP_DEG = 0.5           # Slider resolution; currents are evaluated by the saved polynomial.
WINDOW_S = 10.0                 # Duration of the rolling angle plot, in simulated seconds.
COMMAND_INTERVAL_S = 0.01       # Command/sample interval; physical integration is still 0.0001 s.
FPS = 20                       # Maximum display refresh rate, independent of the physics step.
TIME_SCALE = 1.0                # 1.0 = one simulated second per wall-clock second.
MAX_BLOCKS_PER_FRAME = 10       # Limit catch-up work so slider/buttons remain responsive.
AUTO_START = os.environ.get('SPIKINGNN_NOTEBOOK_SMOKE') != '1'  # Set False to start manually.

FONT_FAMILY = 'Times New Roman' # Font used for all labels inside the figure.
FONT_SIZE = 15                  # Base figure font size.
TITLE_SIZE = 15                 # Panel titles.
LABEL_SIZE = 15                 # Axis labels.
TICK_SIZE = 15                  # Tick labels.
LEGEND_SIZE = 15                # Legend text.
ANNOTATION_SIZE = 11            # Angles, time and force values printed on the diagram.
WIDGET_FONT_SIZE = 15           # Text size of the HTML status/readout, in pixels.
FIGSIZE = (12.5, 5.5)           # Figure size in inches: schematic on the left, traces on the right.
FRAME_DPI = 95                 # Interactive image resolution; lower this if display is slow.
SNAPSHOT_DPI = 220              # Resolution of the PNG saved by the snapshot button.
FORCE_SCALE_M_PER_N = 0.004     # Base display length per newton, before visual length limits.
GRAVITY_MIN_ARROW_M = 0.06      # Minimum visible gravity-arrow length; does not change physical mg.
MAX_ARROW_M = 0.16              # Cap display length only; printed force values remain unchanged.
SNAPSHOT_DIR = None             # None saves under D/results/live_snapshots; no helper files are needed.
FLEXOR_COLOR = '#d62728'         # Flexor muscle: red.
EXTENSOR_COLOR = '#1764c0'       # Extensor muscle: blue.
TARGET_COLOR = '#e6272f'         # Desired direction and desired-angle curve: green.
LIMB_COLOR = '#000000'           # Actual limb and actual-angle curve: dark gray.


def _live_project_root():
    """Locate the project from the notebook directory or repository root."""
    for parent in (Path.cwd(), *Path.cwd().parents):
        for candidate in (parent, parent / 'scripts/AfferentBalance2026'):
            if (candidate / 'experiments/D_trajectory/genome_cfg.json').is_file():
                return candidate.resolve()
    raise FileNotFoundError('Cannot locate scripts/AfferentBalance2026')


_live_project = _live_project_root()
sys.path.insert(0, str(_live_project.parents[1]))
import importlib
from scripts.AfferentBalance2026.experiments import notebook_runtime as _notebook_runtime
# Refresh this helper too: an open kernel may retain its previous implementation.
importlib.reload(_notebook_runtime)
# Update core.genome before importing dependants that require MIRROR_INDICES.
_notebook_runtime.refresh_model_modules()
resolve_genome_path = _notebook_runtime.resolve_genome_path
from scripts.AfferentBalance2026.experiments.D_trajectory.controller import AngleController, load_solution
from scripts.AfferentBalance2026.experiments.simulation import MODEL
if MODEL != "SpikingNN.Afferents/v3-tensile":
    raise RuntimeError("Restart the Jupyter kernel: tensile-torque model v3 is required.")

# Register locally installed fonts without changing the backend of earlier plots.
for _font_dir in (Path.home() / '.local/share/fonts', Path.home() / '.wine/drive_c/windows/Fonts'):
    for _font_path in _font_dir.rglob('times*.ttf'):
        font_manager.fontManager.addfont(str(_font_path))
_live_font = FONT_FAMILY if any(f.name == FONT_FAMILY for f in font_manager.fontManager.ttflist) else 'DejaVu Serif'
_live_genome = resolve_genome_path(
    _live_project / 'experiments/D_trajectory', GENOME_PATH, variant=GENOME_VARIANT,
)
print("Loaded genome:", _live_genome)
_live_snapshots = (Path(SNAPSHOT_DIR).expanduser().resolve() if SNAPSHOT_DIR is not None
                   else _live_project / 'experiments/D_trajectory/results/live_snapshots')


class LivePendulumSimulation:
    """Display a continuously evolving afferent controller with an online angle input.

    The asyncio task yields between frames, allowing Jupyter widget callbacks
    to run. Stop/resume preserves every neural, muscle and mechanical state.
    Muscle arrows show tensile forces along tendons toward the fixed anchors.
    Their generalized moments follow M_i = -F_i*dL_i/dq in the corrected model.
    """

    def __init__(self, genome_path, snapshot_dir):
        """Load the genome, warm up Numba, build controls and draw the reset state."""
        ticks = COMMAND_INTERVAL_S / .0001
        if (not np.isfinite([COMMAND_INTERVAL_S, WINDOW_S, FPS, TIME_SCALE,
                             FORCE_SCALE_M_PER_N, MAX_ARROW_M, GRAVITY_MIN_ARROW_M]).all()
                or COMMAND_INTERVAL_S < .0001 or not np.isclose(ticks, round(ticks), rtol=0, atol=1e-7)
                or min(WINDOW_S, FPS, TIME_SCALE, FORCE_SCALE_M_PER_N, MAX_ARROW_M, GRAVITY_MIN_ARROW_M) <= 0
                or GRAVITY_MIN_ARROW_M > MAX_ARROW_M
                or not isinstance(MAX_BLOCKS_PER_FRAME, int) or MAX_BLOCKS_PER_FRAME < 1):
            raise ValueError('Invalid timing or arrow settings; commands must use the 0.1 ms grid')
        self.genome_path = Path(genome_path)
        self.snapshot_dir = Path(snapshot_dir)
        cfg, physical, _, _, _ = load_solution(self.genome_path)
        self.mass, self.length, _, self.gravity = cfg.mechanics
        self.a1, self.a2 = physical[46:48]
        self.controller = AngleController(self.genome_path, initial_angle_deg=INITIAL_ANGLE_DEG,
                                           initial_velocity_rad_s=INITIAL_SPEED_RAD_S,
                                           target_angle_deg=INITIAL_TARGET_DEG)
        self.controller.advance(INITIAL_TARGET_DEG, COMMAND_INTERVAL_S)  # Compile before timing starts.
        self.controller.reset(INITIAL_ANGLE_DEG, INITIAL_SPEED_RAD_S, INITIAL_TARGET_DEG)
        self.history = deque(maxlen=math.ceil(WINDOW_S / COMMAND_INTERVAL_S) + 3)
        self._task, self.running, self.closed = None, False, False
        self.last_error, self.last_snapshot_path = None, None
        self._record()
        self.target_slider = widgets.FloatSlider(
            value=INITIAL_TARGET_DEG, min=-45., max=45., step=TARGET_STEP_DEG,
            description='Цель, °:', continuous_update=True, readout_format='.1f',
            style={'description_width': 'initial'}, layout=widgets.Layout(width='470px'))
        self.start_stop = widgets.Button(description='Старт', icon='play', button_style='success')
        self.reset_button = widgets.Button(description='Сброс', icon='undo')
        self.snapshot_button = widgets.Button(description='Снимок PNG', icon='camera')
        self.readout = widgets.HTML()
        self.message = widgets.HTML()
        self.image = widgets.Image(format='png', layout=widgets.Layout(width='100%', max_width='1250px'))
        controls = widgets.HBox([self.start_stop, self.reset_button, self.snapshot_button, self.target_slider],
                                 layout=widgets.Layout(flex_flow='row wrap', align_items='center'))
        self.ui = widgets.VBox([controls, self.readout, self.image, self.message])
        self.target_slider.observe(self._target_changed, names='value')
        self.start_stop.on_click(self._toggle)
        self.reset_button.on_click(self._reset_clicked)
        self.snapshot_button.on_click(self._snapshot_clicked)
        self._make_figure()
        self._draw()
        self._notice('Мышечные времена ≥50 мс: загружен переобученный геном. Метрики проверки приведены в README.')

    def _make_figure(self):
        """Create persistent artists once; later frames only update their data."""
        style = {'font.family': _live_font, 'font.size': FONT_SIZE, 'axes.titlesize': TITLE_SIZE,
                 'axes.labelsize': LABEL_SIZE, 'xtick.labelsize': TICK_SIZE,
                 'ytick.labelsize': TICK_SIZE, 'legend.fontsize': LEGEND_SIZE, 'mathtext.fontset': 'stix'}
        with mpl.rc_context(style):
            self.figure = Figure(figsize=FIGSIZE, dpi=FRAME_DPI, layout='constrained')
            self.canvas = FigureCanvasAgg(self.figure)
            self.scene, self.chart = self.figure.subplots(1, 2, gridspec_kw={'width_ratios': [1., 1.35]})
            ax = self.scene
            extent = max(self.length, self.a1 + .06, self.a2 + MAX_ARROW_M) * 1.12
            ax.set(xlim=(-extent, extent), ylim=(-self.length*.65, self.length*1.28),
                   aspect='equal', title='')
            ax.axis('off')
            ax.plot([-extent*.9, extent*.9], [0, 0], color='#a7a7a7', lw=1, zorder=0)
            ax.plot([0, 0], [0, self.length*1.06], ':', color='#b6b6b6', lw=1)
            self.target_line, = ax.plot([], [], '-', color=TARGET_COLOR, lw=2, zorder=6, label='Tartget')
            self.flexor_line, = ax.plot([], [], color=FLEXOR_COLOR, lw=2.5, label='Flexor')
            self.extensor_line, = ax.plot([], [], color=EXTENSOR_COLOR, lw=2.5, label='Extensor')
            self.limb_line, = ax.plot([], [], '-o', color=LIMB_COLOR, lw=4, markersize=5, label='Limb')
            ax.plot([0], [0], 'o', color='black', ms=8, zorder=10)
            ax.plot([self.a1], [0], 's', color=FLEXOR_COLOR, ms=5)
            ax.plot([-self.a1], [0], 's', color=EXTENSOR_COLOR, ms=5)
            self.force_arrows = []
            for color in (FLEXOR_COLOR, EXTENSOR_COLOR, '#9b6818'):
                arrow = FancyArrowPatch((0, 0), (.01, 0), arrowstyle='-|>',
                                         mutation_scale=15, lw=2, color=color, zorder=8)
                ax.add_patch(arrow)
                self.force_arrows.append(arrow)
            self.gravity_label = ax.text(0, 0, 'mg', color='#9b6818', fontsize=ANNOTATION_SIZE)
            self.force_text = ax.text(.01, .17, '', transform=ax.transAxes,
                                      fontsize=ANNOTATION_SIZE, va='bottom')
            self.angle_text = ax.text(.02, .96, '', transform=ax.transAxes,
                                      fontsize=ANNOTATION_SIZE, va='top')
            ax.legend(loc='lower left', bbox_to_anchor=(.0, -.1), ncol=2, framealpha=.85)
            self.target_trace, = self.chart.plot([], [], '--', color=TARGET_COLOR, lw=2,
                                                 drawstyle='steps-post', label='Target angle')
            self.angle_trace, = self.chart.plot([], [], color=LIMB_COLOR, lw=1.5, label='Limb`s angle')
            self.chart.set(title='', xlabel='t, s',
                            ylabel='Angle, °', xlim=(0, WINDOW_S), ylim=(-55, 55))
            # Place bold panel labels in each subplot's upper-left corner.
            for panel, label in ((self.scene, 'A'), (self.chart, 'B')):
                panel.set_title(label, loc='left', fontweight='bold', fontsize=TITLE_SIZE)
            self.chart.grid(alpha=.25)
            self.chart.legend(loc='upper right')
            self.figure.supxlabel('Muscle arrows: tendon tension toward anchors. '
                                   'mg: gravity (enlarged for visibility). Model: v3-tensile.',
                                   fontsize=max(8, ANNOTATION_SIZE-1), fontfamily=_live_font)

    def geometry(self, state=None):
        """Return diagram geometry and force vectors consistent with model moments.

        Positive theta is clockwise from the upward vertical: r=(sin(theta), cos(theta)).
        Tendon lengths match L(q) at q=theta-pi/2. Positive tension pulls the
        attachment toward its anchor; moment is force dotted with dr/dtheta.
        Gravity acts downward at the midpoint of the uniform rod.
        """
        state = self.controller.snapshot() if state is None else state
        theta = np.deg2rad(state['angle_deg'])
        direction = np.array([np.sin(theta), np.cos(theta)])
        tangent = np.array([np.cos(theta), -np.sin(theta)])
        attachment = self.a2*direction
        anchors = np.array([[self.a1, 0.], [-self.a1, 0.]])
        lengths = np.linalg.norm(anchors-attachment, axis=1)
        # A muscle applies tension toward its fixed anchor, never a pushing force.
        muscle_vectors = state['force_N'][:, None]*(anchors-attachment)/lengths[:, None]
        moments = muscle_vectors @ (self.a2*tangent)
        gravity_vector = np.array([0., -self.mass*self.gravity])
        return dict(direction=direction, tangent=tangent, attachment=attachment, anchors=anchors,
                    lengths=lengths, moments=moments, muscle_vectors=muscle_vectors,
                    gravity_vector=gravity_vector, center=.5*self.length*direction)

    def _record(self):
        """Retain the rolling window and one preceding sample for a continuous boundary."""
        s = self.controller.snapshot()
        row = (s['time_s'], s['target_angle_deg'], s['angle_deg'])
        if self.history and abs(self.history[-1][0]-row[0]) < 1e-12:
            self.history[-1] = row
        else:
            self.history.append(row)
        while len(self.history) > 2 and self.history[1][0] < row[0]-WINDOW_S:
            self.history.popleft()

    def _advance_block(self):
        """Advance one command interval; no neuron, muscle or limb state is reset."""
        s = self.controller.advance(float(self.target_slider.value), COMMAND_INTERVAL_S)
        if s['failure'] == 'nonfinite':
            raise FloatingPointError('Нечисловое состояние; требуется сброс.')
        self._record()
        return s

    def _draw_arrow(self, artist, origin, force, min_length=0.):
        """Apply visual length limits while preserving force direction and physical values."""
        delta = FORCE_SCALE_M_PER_N*np.asarray(force)
        length = np.linalg.norm(delta)
        artist.set_visible(length > 1e-10)
        if length > 1e-10:
            delta *= np.clip(length, min_length, MAX_ARROW_M)/length
        artist.set_positions(origin, origin+delta)
        return delta

    def _draw(self):
        """Render the real current state and rolling traces into the notebook image widget."""
        s = self.controller.snapshot()
        if not np.isfinite([s['angle_deg'], s['velocity_rad_s'], *s['force_N']]).all():
            return  # Keep the last valid frame if the numerical model fails.
        g = self.geometry(s)
        target = np.deg2rad(self.target_slider.value)
        # Extend the target slightly so it stays visible when the limb reaches it.
        target_tip = 1.12*self.length*np.array([np.sin(target), np.cos(target)])
        limb_tip = self.length*g['direction']
        self.target_line.set_data([0, target_tip[0]], [0, target_tip[1]])
        self.limb_line.set_data([0, limb_tip[0]], [0, limb_tip[1]])
        for line, anchor in zip((self.flexor_line, self.extensor_line), g['anchors']):
            line.set_data([anchor[0], g['attachment'][0]], [anchor[1], g['attachment'][1]])
        for arrow, force in zip(self.force_arrows[:2], g['muscle_vectors']):
            self._draw_arrow(arrow, g['attachment'], force)
        gravity_delta = self._draw_arrow(
            self.force_arrows[2], g['center'], g['gravity_vector'], min_length=GRAVITY_MIN_ARROW_M)
        self.gravity_label.set_position(g['center']+.5*gravity_delta+np.array([.012, 0.]))
        ff, fe = s['force_N']
        mf, me = g['moments']
        self.force_text.set_text(f'Силы мышц: Fсг = {ff:.1f} Н; Fразг = {fe:.1f} Н\n'
                                 f'Моменты: Mсг = {mf:+.3f} Н·м; Mразг = {me:+.3f} Н·м')
        self.angle_text.set_text(f't = {s["time_s"]:.2f} с\nθ* = {self.target_slider.value:+.1f}°\n'
                                 f'θ = {s["angle_deg"]:+.1f}°')
        data = np.asarray(self.history)
        self.target_trace.set_data(data[:, 0], data[:, 1])
        self.angle_trace.set_data(data[:, 0], data[:, 2])
        right = max(WINDOW_S, s['time_s'])
        self.chart.set_xlim(right-WINDOW_S, right)
        limit = max(55., float(np.max(abs(data[:, 1:]))+5.))
        self.chart.set_ylim(-limit, limit)
        state_label = 'Работает' if self.running else 'Пауза'
        self.readout.value = (f'<div style="font-family:{escape(_live_font)};font-size:{WIDGET_FONT_SIZE}px">'
                              f'<b>{state_label}</b> · t = {s["time_s"]:.2f} с · '
                              f'цель = {self.target_slider.value:+.1f}° · угол = {s["angle_deg"]:+.1f}° · '
                              f'скорость = {s["velocity_rad_s"]:+.2f} рад/с</div>')
        # A PNG widget avoids backend switching and works in JupyterLab and Notebook.
        buffer = BytesIO()
        self.canvas.print_png(buffer)
        self.image.value = buffer.getvalue()

    def _notice(self, text, error=False):
        """Show a compact status or saved-image path, with HTML escaping."""
        color = '#a32020' if error else '#555555'
        self.message.value = (f'<span style="color:{color};font-size:{WIDGET_FONT_SIZE}px">'
                               f'{escape(str(text))}</span>')

    def _target_changed(self, change):
        """Update the command immediately while preserving the actual limb state."""
        self.controller.target_angle_deg = float(change['new'])
        self._record()
        if not self.running:
            self._draw()

    def _toggle(self, _button):
        """Handle the combined start/stop button."""
        self.stop() if self.running else self.start()

    def start(self):
        """Start or resume without resetting state or counting time spent paused."""
        if self.closed or self.running:
            return
        if self.controller.snapshot()['failure'] != 'none':
            self._notice('Расчёт остановлен после падения. Нажмите «Сброс».', error=True)
            return
        self.running = True
        self._wall_origin = time.perf_counter()
        self._sim_origin = self.controller.snapshot()['time_s']
        self.start_stop.description, self.start_stop.icon, self.start_stop.button_style = 'Стоп', 'pause', 'warning'
        self._notice('Меняйте целевой угол ползунком. Снимок сохраняет оба графика и ставит расчёт на паузу.')
        self._task = asyncio.get_running_loop().create_task(self._run())

    async def _run(self):
        """Track wall time with bounded catch-up and yield regularly to widget callbacks."""
        task = asyncio.current_task()
        try:
            while self.running and self._task is task:
                started = time.perf_counter()
                desired_elapsed = (started-self._wall_origin)*TIME_SCALE
                actual_elapsed = self.controller.snapshot()['time_s']-self._sim_origin
                blocks = min(MAX_BLOCKS_PER_FRAME,
                             max(0, int((desired_elapsed-actual_elapsed+1e-10)/COMMAND_INTERVAL_S)))
                for _ in range(blocks):
                    s = self._advance_block()
                    if s['failure'] != 'none':
                        self.running = False
                        self._notice('Конечность вышла за ±90°. Расчёт остановлен; нажмите «Сброс».', error=True)
                        break
                self._draw()
                # Every 0.1 ms physical step is retained. Slow rendering can cause lag,
                # but elapsed simulation time is never fabricated or skipped forward.
                await asyncio.sleep(max(.001, 1/FPS-(time.perf_counter()-started)))
        except asyncio.CancelledError:
            pass
        except Exception as error:
            self.last_error = error
            self._notice(f'Расчёт остановлен: {error}', error=True)
        finally:
            if self._task is task:
                self._task, self.running = None, False
                self.start_stop.description, self.start_stop.icon, self.start_stop.button_style = 'Старт', 'play', 'success'
                self._draw()

    def stop(self):
        """Freeze the model and chart; the next start resumes from this exact state."""
        self.running = False
        task, self._task = self._task, None
        if task is not None and not task.done():
            task.cancel()
        self.start_stop.description, self.start_stop.icon, self.start_stop.button_style = 'Старт', 'play', 'success'
        if not self.closed:
            self._draw()

    def _reset_clicked(self, _button):
        """Reset the physical state and history explicitly, retaining the current slider target."""
        self.stop()
        self.controller.reset(INITIAL_ANGLE_DEG, INITIAL_SPEED_RAD_S, float(self.target_slider.value))
        self.history.clear()
        self.last_error = None
        self._record()
        self._draw()
        self._notice('Начальное состояние восстановлено. Нажмите «Старт».')

    def save_snapshot(self):
        """Pause and save a high-resolution PNG of the schematic and the 10 s chart."""
        self.stop()
        self.snapshot_dir.mkdir(parents=True, exist_ok=True)
        stamp = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
        path = self.snapshot_dir / f'pendulum_{stamp}_t{self.controller.snapshot()["time_s"]:.2f}s.png'
        self.figure.savefig(path, dpi=SNAPSHOT_DPI, bbox_inches='tight')
        self.last_snapshot_path = path
        self._notice(f'Пауза. Снимок сохранён: {path}')
        return path

    def _snapshot_clicked(self, _button):
        """Report snapshot errors in the widget instead of losing the button callback."""
        try:
            self.save_snapshot()
        except Exception as error:
            self._notice(f'Не удалось сохранить снимок: {error}', error=True)

    def close(self):
        """Cancel the background task and release this cell's old widget instance."""
        if self.closed:
            return
        self.stop()
        self.closed = True
        self.target_slider.unobserve(self._target_changed, names='value')
        for widget in (self.target_slider, self.start_stop, self.reset_button, self.snapshot_button,
                       self.readout, self.message, self.image, self.ui):
            widget.close()
        self.figure.clear()


# Rerunning the cell replaces its previous animation instead of starting another task.
_previous_live_pendulum = globals().get('live_pendulum')
if _previous_live_pendulum is not None:
    _previous_live_pendulum.close()
live_pendulum = LivePendulumSimulation(_live_genome, _live_snapshots)
display(live_pendulum.ui)
if AUTO_START:
    live_pendulum.start()
# Optional programmatic controls: live_pendulum.stop(), .start(), .save_snapshot().
